# 5.14 模型不動或震盪要怎麼找步幅？


訓練loss幾乎不動，可能步子太小；跳上跳下甚至越來越大，可能步子太大。但同樣現象也可能來自資料與流程錯誤，因此先按 [5.1](https://birdhackor.github.io/tiny-perceptron-vlm/5.1.html) 確認固定小題會學，再在固定模型、數據與優化器下試幾種學習率。一次只改步幅，才知道曲線變化來自哪裡。

[1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html) 已示範梯度反方向不保證大步有效。本節把幾個候選放在一起，用代價 `(w-3)²`、初始w=1、梯度-4。每種候選都從相同起點走一小步，而不是讓第一個候選走完，再把它的終點當第二個起點。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

w = torch.tensor(1.0)
gradient = 2 * (w - 3)
print("起點代價", (w - 3).square().item())
for lr in [0.01, 0.1, 2.0]:
    new = w - lr * gradient
    print("步幅", lr, "新位置", new.item(), "新代價", (new - 3).square().item())

輸入起點與三種學習率，代碼每輪只計算new，沒有覆蓋w，因此比較條件一致。0.01到1.04，代價約3.8416，改善很少；0.1到1.4，代價2.56；2.0到9，代價36，遠比原來4差。大步雖沿正確初始方向，卻跨過目標3走上另一邊。這個具體反例比「大學習率不穩」一句話更能解釋發生什麼。

對真正模型可用小份固定資料做學習率範圍試驗，觀察前幾步loss是否下降、是否突然跳大、梯度有沒有異常。第一次先試量級，例如0.0001、0.001、0.01，再在有效範圍細分；不要只根據一次幸運下降就定案。各次試驗從同一組初始參數與optimizer歷史開始，使用同一資料順序及同樣更新步數，在同一步數觀察loss；最後參數與loss是要比較和記錄的結果，無需相同。種子是讓隨機選擇可重現的起始設定，應使用相同設定，見[5.15](https://birdhackor.github.io/tiny-perceptron-vlm/5.15.html)。

若步幅很小也完全沒有變化，先看參數前後是否真的移動：optimizer是否拿到對的參數、step有沒有執行、答案有沒有被全部忽略。若loss出現非有限值，例如無窮或NaN，也要查分母、輸入數值與求導，而不是無限縮小學習率掩蓋原因。這裡NaN意為不是可用數字，常見於未定義運算。

本節只驗證一個平方函數的候選比較，不能把0.1直接當作任何語言模型的最佳學習率。模型梯度維度多、Adam還帶歷史、batch也會影響波動；batch就是一次一起處理的資料組，這組題目換了，更新方向也可能不同。如果正式腳本使用 `--lr`，應在記錄中說明實際數值與排程；排程是學習率隨更新進度改變的規則，例如前幾步漸增、之後漸減，不能只寫「使用預設」。

練習只把起點w改成4，先手算梯度2、三種新位置為3.98、3.8、0，代價約0.9604、0.64、9，再執行核對。改起點會改梯度與絕對代價，但三個候選的新代價除以起點代價，仍分別約為0.9604、0.64、9；這個平方函數在兩個起點用0.5都能一步到3。因此這次練習沒有證明最佳步幅隨起點改變，只再次說明沿正確梯度方向走，仍不能保證任意大步幅有效。
